# 6.1 Klasifikasi Penutup Lahan Jawa Timur (Sentinel-2A, 6 Kelas)

**Supervised learning** (berbeda dari clustering Bab 5 yang unsupervised): model belajar dari label poligon sampel.

## Tujuan analisis (land use & land classification)

1. Menyusun dataset training 6 kelas penutup/penggunaan lahan se-Jawa Timur dari sumber resmi (bukan digitasi manual, bukan pseudo-label clustering).
2. Mendeskripsikan kanal (band) Sentinel-2A yang dipakai sebagai fitur.
3. Mengekstraksi fitur spektral + indeks vegetasi/air/bangunan per poligon.
4. Melatih dan membandingkan beberapa model klasifikasi, memilih akurasi tertinggi.
5. Memetakan hasil klasifikasi di atas citra satelit (Google Satellite via Folium/Leaflet, bisa digeser-zoom).
6. Menyajikan sistem informasi klasifikasi via Streamlit + dokumentasi proses di web statis (Jupyter Book ini).

**Wilayah:** Provinsi Jawa Timur (~47.996 km²). **Citra:** Sentinel-2A Level-2A (surface reflectance), komposit median 2025-08-01 s.d. 2025-10-15 (musim kemarau, minim awan).

In [1]:
import os
import geopandas as gpd
import pandas as pd

SEED = 42
GPKG = "jatim/training_samples_jawa_timur.gpkg"
LAYER = "training_samples"
BOUNDARY = "jatim/boundary_jatim.geojson"

g = gpd.read_file(GPKG, layer=LAYER)
print("CRS:", g.crs, "| baris:", len(g), "| kolom:", list(g.columns))
print("geometri kosong/None:", int(g.geometry.is_empty.sum() + g.geometry.isna().sum()))
print("geometri invalid:", int((~g.geometry.is_valid).sum()))
print("\nSebaran poligon per kelas (sebelum split):")
print(g.groupby(["class_id", "class_name"]).size().to_string())

CRS: EPSG:4326 | baris: 2823 | kolom: ['class_id', 'class_name', 'source', 'source_class', 'geometry']
geometri kosong/None: 0


geometri invalid: 0

Sebaran poligon per kelas (sebelum split):
class_id  class_name           
1         Sawah                    500
2         Bangunan / Permukiman    500
3         Mangrove                 500
4         Lahan hijau              500
5         Laut                     500
6         Danau                    323


## Data understanding: 6 kelas + sumber + acuan Indonesia

| class_id | Kelas | Sumber poligon | Kategori sumber | Jumlah poligon |
|---:|---|---|---|---:|
| 1 | Sawah | Kementan (LBS 2019) | per kabupaten/kota (33 kab/kota) | 500 |
| 2 | Bangunan / Permukiman | BIG (Penggunaan Tanah 50K) | Kampung/Perumahan (6 kategori) | 500 |
| 3 | Mangrove | BIG (Peta Mangrove) | Lebat / Sedang / Jarang | 500 |
| 4 | Lahan hijau | BIG (Penggunaan Tanah 50K) | Hutan, kebun, tegalan, semak, padang rumput, ... (13 kategori) | 500 |
| 5 | Laut | Natural Earth 10m ocean | grid lepas pantai (>1 km dari garis pantai, anti mixed-pixel) | 500 |
| 6 | Danau | BIG (kologi Waduk sbg proksi) | Waduk (catatan: waduk ≠ danau alami) | 323 |

**Total: 2.823 poligon.** Kelas 6 paling sedikit (93 fitur sumber → 323 setelah cleaning) — dilaporkan apa adanya, bukan dikarang.

### Acuan klasifikasi lahan di Indonesia (bukan sekadar sawah vs non-sawah)

- **SNI 7645-1:2014** — Klasifikasi Penutup Lahan: sawah, permukiman, hutan, perkebunan, semak, lahan terbuka, perairan, dsb.
- **BIG / Informasi Geospasial Dasar (IGD)** — Peta Penggunaan Tanah skala 50K dan Peta Mangrove sebagai sumber label kelas 2–4 dan 6.
- **Kementan LBS (Lahan Baku Sawah) 2019** — sumber label sawah kelas 1.
- **KLHK (penutupan lahan)** — hutan primer/sekunder, perkebunan, pertanian lahan kering sebagai pembanding definisi "lahan hijau".

Pemetaan ke 6 kelas UTS: Sawah → SNI sawah; Bangunan/Permukiman → SNI permukiman + lahan terbangun; Mangrove → SNI hutan mangrove; Lahan hijau → hutan/kebun/tegalan/semak; Laut → perairan laut; Danau → perairan danau/waduk.

## Koleksi data & band Sentinel-2A

Citra diunduh via **openEO (Copernicus Data Space)**, koleksi `SENTINEL2_L2A`, komposit **median** per piksel (menekan awan/bayangan) — lihat `download_raster_jatim.py` (reproducible, fase `auth → features → demo`).

### Macam-macam band Sentinel-2A (13 band)

| Band | Nama | Panjang gelombang | Resolusi | Kegunaan untuk klasifikasi |
|---|---|---|---|---|
| B01 | Coastal aerosol | 443 nm | 60 m | koreksi atmosfer (tidak dipakai fitur) |
| **B02** | **Blue** | 490 nm | 10 m | ✔ fitur; tanah/air vs vegetasi, komponen EVI |
| **B03** | **Green** | 560 nm | 10 m | ✔ fitur; puncak reflektansi vegetasi, komponen NDWI |
| **B04** | **Red** | 665 nm | 10 m | ✔ fitur; absorbsi klorofil, komponen NDVI/EVI/SAVI |
| B05 | Red-edge 1 | 705 nm | 20 m | transisi red→NIR (kesehatan vegetasi) |
| B06 | Red-edge 2 | 740 nm | 20 m | idem |
| B07 | Red-edge 3 | 783 nm | 20 m | idem |
| **B08** | **NIR** | 842 nm | 10 m | ✔ fitur; biomassa vegetasi, komponen NDVI/NDWI/NDBI/NDMI |
| B8A | NIR narrow | 865 nm | 20 m | red-edge/NIR sempit |
| B09 | Water vapour | 945 nm | 60 m | koreksi atmosfer (tidak dipakai fitur) |
| B10 | Cirrus | 1375 nm | 60 m | deteksi awan tipis (tidak dipakai fitur) |
| **B11** | **SWIR-1** | 1610 nm | 20 m | ✔ fitur; lahan terbangun/tanah terbuka, komponen NDBI/NDMI |
| **B12** | **SWIR-2** | 2190 nm | 20 m | ✔ fitur; mineralogi tanah, kebakaran |

**6 band fitur (digarisbawahi di fase ekstraksi): B02, B03, B04, B08, B11, B12.** Band 60 m (B01/B09/B10) dan red-edge (B05–B07/B8A) tidak dipakai agar fitur konsisten dan ringan.

## Fitur yang diekstraksi + rumus

Per poligon diambil **nilai mean tiap band** (piksel nodata dikecualikan), lalu dihitung 6 indeks spektral. Total **12 fitur** per sampel:

| # | Fitur | Rumus | Menangkap |
|---|---|---|---|
| 1–6 | `mean_B02/B03/B04/B08/B11/B12` | $\bar{\rho} = \frac{1}{n}\sum_{i=1}^{n}\rho_i$ | tanda tangan spektral tiap kelas |
| 7 | NDVI | $(B08-B04)/(B08+B04)$ | kerapatan vegetasi (sawah, mangrove, lahan hijau tinggi) |
| 8 | NDWI (McFeeters) | $(B03-B08)/(B03+B08)$ | badan air (laut, danau positif tinggi) |
| 9 | NDBI | $(B11-B08)/(B11+B08)$ | lahan terbangun (bangunan positif) |
| 10 | NDMI | $(B08-B11)/(B08+B11)$ | kelembapan vegetasi |
| 11 | EVI | $2{,}5\cdot(B08-B04)/(B08+6\cdot B04-7{,}5\cdot B02+1)$ | vegetasi tanpa saturasi + koreksi atmosfer |
| 12 | SAVI | $((B08-B04)/(B08+B04+0{,}5))\cdot1{,}5$ | vegetasi dengan koreksi brightness tanah |

Intuisi UTS: **laut/danau** → NDWI tinggi, NDVI negatif; **bangunan** → NDBI positif; **sawah/mangrove/lahan hijau** → NDVI/EVI tinggi (dipisah via SWIR + tekstur band).

In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# EDA cepat: luas per kelas (m2, CRS UTM 49S)
gm = g.to_crs("EPSG:32749").copy()
gm["area_m2"] = gm.geometry.area
stat = gm.groupby(["class_id", "class_name"]).agg(
    n=("geometry", "size"),
    total_ha=("area_m2", lambda a: a.sum() / 1e4),
    median_m2=("area_m2", "median"),
).reset_index().sort_values("class_id")
print(stat.round(1).to_string(index=False))
print("Kabupaten/kota unik (sawah):", gm[gm.class_id == 1]["source_class"].nunique())

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(stat["class_name"], stat["n"])
ax.set_title("Jumlah poligon training per kelas (total 2823)")
ax.set_ylabel("poligon")
fig.autofmt_xdate(rotation=20)
fig.tight_layout()
print("grafik: OK")

# Rencana split stratified 80/20 (dieksekusi penuh di 6.2 setelah fitur raster tiba)
idx = gm.index.to_numpy()
y = gm["class_id"].to_numpy()
tr, te = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=y)
print("\nRencana split 80/20 (stratified, seed 42):")
print(pd.Series(y[pd.Index(idx).get_indexer(tr)]).value_counts().sort_index().to_dict(), "-> train:", len(tr))
print(pd.Series(y[pd.Index(idx).get_indexer(te)]).value_counts().sort_index().to_dict(), "-> test:", len(te))

 class_id            class_name   n  total_ha  median_m2
        1                 Sawah 500    8587.7    20268.9
        2 Bangunan / Permukiman 500    1933.9     9405.0
        3              Mangrove 500    2137.5     1976.1
        4           Lahan hijau 500  132160.7    15837.8
        5                  Laut 500 1252883.7 25056562.0
        6                 Danau 323    2390.8     3485.0
Kabupaten/kota unik (sawah): 33


grafik: OK



Rencana split 80/20 (stratified, seed 42):
{1: 400, 2: 400, 3: 400, 4: 400, 5: 400, 6: 258} -> train: 2258
{1: 100, 2: 100, 3: 100, 4: 100, 5: 100, 6: 65} -> test: 565


## Peta interaktif: sebaran sampel + Google Satellite (Folium)

Peta di bawah memakai **Folium/Leaflet**: bisa **digeser, zoom, dan toggle layer per kelas**, dengan basemap **Google Satellite** (+ OpenStreetMap). Setiap kelas digambar sebagai **poligon penuh berwarna** (geometri disederhanakan ±50 m agar halaman tetap ringan; bedanya tak terlihat di zoom provinsi). Versi layar-penuh: `maps/folium_jatim_samples.html`.

In [3]:
import folium
from folium import GeoJsonTooltip

COLORS = {1: ("#1b5e20", "#2e7d32"), 2: ("#b71c1c", "#e53935"), 3: ("#004d40", "#00695c"),
          4: ("#33691e", "#9ccc65"), 5: ("#0d47a1", "#1565c0"), 6: ("#00838f", "#4dd0e1")}
NAMES = dict(g.drop_duplicates("class_id").set_index("class_id")["class_name"])

# Sederhanakan geometri ±50 m (halaman tetap ringan, tampilan identik di zoom provinsi)
gs = g.copy()
gs["geometry"] = g.to_crs("EPSG:32749").geometry.simplify(50).to_crs("EPSG:4326")
print("ukuran geojson sederhana:", round(len(gs.to_json()) / 1e6, 2), "MB")

b = g.total_bounds  # [west, south, east, north]
map_center = [(b[1] + b[3]) / 2, (b[0] + b[2]) / 2]
m = folium.Map(location=map_center, zoom_start=8,
               tiles=None, control_scale=True)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google Satellite", name="Google Satellite", overlay=False, max_zoom=20,
).add_to(m)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", overlay=False).add_to(m)
folium.GeoJson(open(BOUNDARY, encoding="utf-8").read(), name="Batas Jatim",
               style_function=lambda _: {"color": "yellow", "weight": 2, "fillOpacity": 0}).add_to(m)
for cid in sorted(g["class_id"].unique().tolist()):
    edge, fill = COLORS[cid]
    fg = folium.FeatureGroup(name=f"Kelas {cid}: {NAMES[cid]}")
    folium.GeoJson(gs[gs["class_id"] == cid].to_json(),
                   style_function=lambda _, e=edge, f=fill: {
                       "color": e, "weight": 1.2, "fillColor": f, "fillOpacity": 0.55},
                   tooltip=GeoJsonTooltip(fields=["class_name", "source", "source_class"],
                                          aliases=["Kelas", "Sumber", "Kategori"],
                                          sticky=True)).add_to(fg)
    fg.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save("jatim/folium_jatim_samples.html")
print("tersimpan: jatim/folium_jatim_samples.html |", {NAMES[c]: int((g.class_id == c).sum()) for c in sorted(NAMES)})
m

ukuran geojson sederhana: 1.77 MB


tersimpan: jatim/folium_jatim_samples.html | {'Sawah': 500, 'Bangunan / Permukiman': 500, 'Mangrove': 500, 'Lahan hijau': 500, 'Laut': 500, 'Danau': 323}


## Status & lanjut ke 6.2

- ✅ Data training poligon: **lengkap (2.823, 6 kelas)** + boundary Jatim.
- ✅ Data raster Sentinel-2A: `jatim/features/*.csv` + `jatim/raster/s2_demo_sidoarjo.tif` (via `download_raster_jatim.py`).
- ▶ **6.2**: ekstraksi fitur → split 80/20 → banding model (RandomForest, SVM, KNN, GradientBoosting) → pilih akurasi tertinggi → metrik + peta jendela demo.
- ▶ **6.3**: peta hasil klasifikasi jendela demo + Folium Google Satellite.
- ▶ **6.4**: aplikasi Streamlit (`streamlit_jatim.py`) + deploy.